# Evaluating retrieval: worked solution

Worked solution for the exercise in the lesson **[Evaluating retrieval](https://ramkumarmd.com/medical-ai/rag/retrieval-evaluation)** (*Retrieval-augmented generation (RAG)* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](04-retrieval-evaluation.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** If retrieval fails, the model never sees the right evidence, so evaluate retrieval separately from answers. Build a set of realistic questions labelled with the passages that answer them, then measure how often the right passage appears in the top results (recall@k) and how high it ranks (mean reciprocal rank). Compare methods on the same set, and read every miss: small test sets are noisy, and the misses tell you what to fix.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Setup

In [ ]:
%pip install -q sentence-transformers

## Code from earlier lessons

This notebook builds on code from earlier lessons. Run these cells first.

From **[Chunking clinical guidelines](https://ramkumarmd.com/medical-ai/rag/chunking-clinical-guidelines)**: A synthetic clinic protocol with five headed sections.

In [ ]:
PROTOCOL = """# Synthetic diabetes clinic protocol (teaching document, not clinical guidance)

## 1. Monitoring glucose control
Check HbA1c every 3 months after any change in treatment. Once targets are met and treatment is stable, check HbA1c every 6 months. A common target for many non-pregnant adults is below 7%, adjusted for the individual.

## 2. Kidney function and metformin
Check eGFR and urine albumin-to-creatinine ratio at least once a year. Metformin should not be started if eGFR is below 45. If eGFR falls below 30 during treatment, stop metformin.

## 3. Foot care
Examine both feet at every review. Test sensation with a 10 g monofilament at least once a year. Refer urgently if there is an ulcer with spreading redness, a suspected infection of the bone, or a cold, pale foot.

## 4. Low blood glucose
A glucose below 70 mg/dL is low. Treat with 15 g of fast-acting carbohydrate and recheck after 15 minutes. Repeat if glucose is still below 70 mg/dL. Review any medicine that may have caused the low glucose.

## 5. Eye screening
Arrange retinal screening at diagnosis of type 2 diabetes and then at the interval set by the local screening programme.
"""

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Scoring three retrieval methods

Run the protocol block from the chunking lesson first. Twelve labelled questions; TF-IDF and embedding scores.

In [ ]:
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

sections = [p.strip() for p in re.split(r"\n(?=## )", PROTOCOL)[1:]]
ids = ["glucose", "kidney", "foot", "hypo", "eye"]

tests = [   # (question, id of the section that answers it)
    ("How often should HbA1c be checked after changing treatment?", "glucose"),
    ("What HbA1c target suits most adults?", "glucose"),
    ("My sugar dropped really low, what now?", "hypo"),
    ("What should I do about a hypo?", "hypo"),
    ("How many grams of carbohydrate treat a low?", "hypo"),
    ("Is the diabetes pill safe with weak kidneys?", "kidney"),
    ("When do we need to stop the biguanide?", "kidney"),
    ("How often is urine albumin checked?", "kidney"),
    ("Numb toes test", "foot"),
    ("When is a foot ulcer an emergency?", "foot"),
    ("How often are retinal photos needed?", "eye"),
    ("When should eye screening start?", "eye"),
]
questions = [q for q, _ in tests]
relevant = [ids.index(r) for _, r in tests]

vec = TfidfVectorizer(stop_words="english").fit(sections + questions)
keyword = cosine_similarity(vec.transform(questions), vec.transform(sections))
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embedding = (model.encode(questions, normalize_embeddings=True)
             @ model.encode(sections, normalize_embeddings=True).T)

### Recall@k and MRR

Continuing from the previous block: metrics for keyword, embedding and a hybrid of both.

In [ ]:
def ranks(scores):
    """1-based rank of the correct section for each question."""
    order = np.argsort(-scores, axis=1)
    return np.array([list(order[i]).index(relevant[i]) + 1 for i in range(len(relevant))])

def report(name, scores):
    r = ranks(scores)
    print(f"{name:<10} recall@1 {np.mean(r <= 1):.2f}   recall@3 {np.mean(r <= 3):.2f}   "
          f"MRR {np.mean(1 / r):.2f}")

def reciprocal_rank_fusion(*score_sets, k=60):
    """Hybrid search: add 1 / (k + rank) from each method."""
    fused = np.zeros_like(score_sets[0])
    for s in score_sets:
        order = np.argsort(-s, axis=1)
        for i in range(s.shape[0]):
            for rank, j in enumerate(order[i], start=1):
                fused[i, j] += 1 / (k + rank)
    return fused

report("Keyword", keyword)
report("Embedding", embedding)
report("Hybrid", reciprocal_rank_fusion(keyword, embedding))

# Keyword    recall@1 0.83   recall@3 0.92   MRR 0.88
# Embedding  recall@1 0.92   recall@3 1.00   MRR 0.94
# Hybrid     recall@1 0.92   recall@3 0.92   MRR 0.94

### Reading the misses

Continuing from the previous block: which questions each method failed to rank first.

In [ ]:
for name, s in [("Keyword", keyword), ("Embedding", embedding),
                ("Hybrid", reciprocal_rank_fusion(keyword, embedding))]:
    r = ranks(s)
    misses = [(questions[i], int(r[i])) for i in np.where(r > 1)[0]]
    print(name, "misses:", misses)

# Keyword misses: [('What should I do about a hypo?', 4), ('Is the diabetes pill safe with weak kidneys?', 3)]
# Embedding misses: [('What should I do about a hypo?', 3)]
# Hybrid misses: [('What should I do about a hypo?', 4)]

## Exercise

**Goal:** Build and use a retrieval test set to choose and improve a retrieval method.

**Data:** Synthetic protocol and labelled questions in the notebook (Created for this course; not clinical guidance)

### Step 1

Reproduce the three methods' recall@1, recall@3 and MRR.

In [ ]:
report("Keyword", keyword)
report("Embedding", embedding)
report("Hybrid", reciprocal_rank_fusion(keyword, embedding))

### Step 2

Add twelve more questions, including three the protocol cannot answer, and decide how to score 'no answer' questions.

In [ ]:
more_tests = [
    ("When does HbA1c move to every 6 months?", "glucose"),
    ("Is 7% the HbA1c goal for everyone?", "glucose"),
    ("What eGFR stops metformin being started?", "kidney"),
    ("Do I keep taking metformin if my kidney function falls to 25?", "kidney"),
    ("Which monofilament weight is used?", "foot"),
    ("What does a cold pale foot mean for referral?", "foot"),
    ("What glucose level counts as low?", "hypo"),
    ("How long should I wait before rechecking after juice for a low?", "hypo"),
    ("Who sets the eye screening interval?", "eye"),
    ("What insulin dose should a new patient start on?", None),   # not in the protocol
    ("Can I drive after a hypo?", None),
    ("Which vaccines do people with diabetes need?", None),
]
all_tests = tests + more_tests
all_q = [q for q, _ in all_tests]
answerable = np.array([r is not None for _, r in all_tests])
rel = np.array([ids.index(r) if r else -1 for _, r in all_tests])

vec2 = TfidfVectorizer(stop_words="english").fit(sections + all_q)
section_emb = model.encode(sections, normalize_embeddings=True)
methods = {
    "Keyword": cosine_similarity(vec2.transform(all_q), vec2.transform(sections)),
    "Embedding": model.encode(all_q, normalize_embeddings=True) @ section_emb.T,
}
methods["Hybrid"] = reciprocal_rank_fusion(methods["Keyword"], methods["Embedding"])

def rank_of_relevant(scores, rel_idx):
    order = np.argsort(-scores, axis=1)
    return np.array([list(order[i]).index(rel_idx[i]) + 1 for i in range(len(rel_idx))])

THRESHOLD = 0.35   # embedding similarity below which the assistant should say "not covered"
for name, s in methods.items():
    r = rank_of_relevant(s[answerable], rel[answerable])
    line = f"{name:<10} answerable ({answerable.sum()}): recall@1 {np.mean(r <= 1):.2f}  recall@3 {np.mean(r <= 3):.2f}  MRR {np.mean(1 / r):.2f}"
    if name == "Embedding":
        declined = (s[~answerable].max(axis=1) < THRESHOLD).mean()
        line += f"   no-answer questions declined at {THRESHOLD}: {declined:.0%}"
    print(line)

Questions the documents cannot answer have no correct section, so they cannot be scored with recall or MRR.
Score them separately as a decision: did the system decline (top similarity below a threshold)? A test set
without such questions rewards a retriever for always returning something, which is exactly the behaviour
that leads to confident wrong answers. Here the embedding threshold of 0.35 declines two of the three
out-of-scope questions; the third scores above it. Hybrid scores are rank-based and have no natural threshold, so the
decline decision here uses the embedding scores.

### Step 3

Apply the abbreviation expansion from the embeddings lesson and re-score all methods.

In [ ]:
ABBREVIATIONS = {"hypo": "hypo (hypoglycaemia, low blood glucose)", "biguanide": "biguanide (metformin)"}

def expand(q):
    for short, long in ABBREVIATIONS.items():
        q = re.sub(rf"\b{short}\b", long, q, flags=re.IGNORECASE)
    return q

expanded_q = [expand(q) for q in all_q]
vec3 = TfidfVectorizer(stop_words="english").fit(sections + expanded_q)
expanded = {
    "Keyword": cosine_similarity(vec3.transform(expanded_q), vec3.transform(sections)),
    "Embedding": model.encode(expanded_q, normalize_embeddings=True) @ section_emb.T,
}
expanded["Hybrid"] = reciprocal_rank_fusion(expanded["Keyword"], expanded["Embedding"])
for name in methods:
    before = rank_of_relevant(methods[name][answerable], rel[answerable])
    after = rank_of_relevant(expanded[name][answerable], rel[answerable])
    print(f"{name:<10} recall@1 {np.mean(before <= 1):.2f} -> {np.mean(after <= 1):.2f}   MRR {np.mean(1 / before):.2f} -> {np.mean(1 / after):.2f}")

### Step 4

Bootstrap the question set (resample with replacement 1,000 times) to get confidence intervals for recall@1 and see whether the methods really differ.

In [ ]:
rng = np.random.default_rng(0)
hits = {name: rank_of_relevant(expanded[name][answerable], rel[answerable]) <= 1 for name in expanded}
n = answerable.sum()
boot = {name: [] for name in hits}
diff = []
for _ in range(1000):
    idx = rng.integers(0, n, n)                       # resample questions with replacement
    for name in hits:
        boot[name].append(hits[name][idx].mean())
    diff.append(hits["Embedding"][idx].mean() - hits["Keyword"][idx].mean())
for name, values in boot.items():
    lo, hi = np.percentile(values, [2.5, 97.5])
    print(f"{name:<10} recall@1 {hits[name].mean():.2f} (95% bootstrap CI {lo:.2f} to {hi:.2f})")
lo, hi = np.percentile(diff, [2.5, 97.5])
print(f"Embedding minus keyword: {np.mean(diff):+.2f} (95% CI {lo:+.2f} to {hi:+.2f})")

def wilson(k, n, z=1.96):
    p = k / n
    centre, half = (p + z**2 / (2 * n)) / (1 + z**2 / n), z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centre - half, centre + half

for name in hits:
    lo, hi = wilson(hits[name].sum(), n)
    print(f"{name:<10} {hits[name].sum()}/{n} correct at rank 1: Wilson 95% CI {lo:.2f} to {hi:.2f}")

After abbreviation expansion the embedding and hybrid methods get all 21 questions right, and the bootstrap
then gives a zero-width interval (1.00 to 1.00): every resample is also perfect. That is a known weakness of the
bootstrap with small samples and extreme results, not evidence of perfection. The Wilson interval shows the
real uncertainty: 21 out of 21 is compatible with a true recall@1 as low as about 0.85. The paired difference
from keyword search has a lower limit of zero. Twenty questions cannot tell similar methods apart; a test set
for choosing a retriever needs many more, drawn from real user questions.

### Step 5

Implement nDCG@3 with graded labels (2 = answers fully, 1 = partly relevant) and compare rankings.

In [ ]:
# Graded labels: 2 = answers the question fully, 1 = partly relevant. Unlisted sections count as 0.
grades = {
    "How often should HbA1c be checked after changing treatment?": {"glucose": 2},
    "What HbA1c target suits most adults?": {"glucose": 2},
    "My sugar dropped really low, what now?": {"hypo": 2},
    "What should I do about a hypo?": {"hypo": 2},
    "How many grams of carbohydrate treat a low?": {"hypo": 2},
    "Is the diabetes pill safe with weak kidneys?": {"kidney": 2},
    "When do we need to stop the biguanide?": {"kidney": 2},
    "How often is urine albumin checked?": {"kidney": 2},
    "Numb toes test": {"foot": 2},
    "When is a foot ulcer an emergency?": {"foot": 2},
    "How often are retinal photos needed?": {"eye": 2},
    "When should eye screening start?": {"eye": 2},
    "When does HbA1c move to every 6 months?": {"glucose": 2},
    "Is 7% the HbA1c goal for everyone?": {"glucose": 2},
    "What eGFR stops metformin being started?": {"kidney": 2},
    "Do I keep taking metformin if my kidney function falls to 25?": {"kidney": 2, "glucose": 1},
    "Which monofilament weight is used?": {"foot": 2},
    "What does a cold pale foot mean for referral?": {"foot": 2},
    "What glucose level counts as low?": {"hypo": 2, "glucose": 1},
    "How long should I wait before rechecking after juice for a low?": {"hypo": 2},
    "Who sets the eye screening interval?": {"eye": 2},
}

def ndcg_at_3(scores, questions_):
    values = []
    for s, q in zip(scores, questions_):
        g = grades[q]
        top = np.argsort(-s)[:3]
        dcg = sum(g.get(ids[j], 0) / np.log2(pos + 2) for pos, j in enumerate(top))
        ideal = sorted(g.values(), reverse=True)[:3]
        idcg = sum(v / np.log2(pos + 2) for pos, v in enumerate(ideal))
        values.append(dcg / idcg)
    return np.mean(values)

ans_q = [q for q, a in zip(all_q, answerable) if a]
for name in expanded:
    print(f"{name:<10} nDCG@3 {ndcg_at_3(expanded[name][answerable], ans_q):.3f}   "
          f"MRR {np.mean(1 / rank_of_relevant(expanded[name][answerable], rel[answerable])):.3f}")

nDCG gives credit for putting the *most* useful section first and some credit for partly relevant sections
lower down; MRR only looks at the first fully relevant one. With almost every question having one relevant
section, the two agree closely here. Graded labels matter more for real guidelines, where several sections
often answer parts of a question.

---

Lesson: [https://ramkumarmd.com/medical-ai/rag/retrieval-evaluation](https://ramkumarmd.com/medical-ai/rag/retrieval-evaluation). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.